Multi-Model Comparison for Pothole Detection in Visually Impaired Pedestrian Navigation
Compares YOLOv8m, YOLOv10m, YOLOv11m, Faster R-CNN, and SSD-VGG16 on a merged pothole dataset

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR, OUTPUT_DIR accordingly so paths work across environments.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB  # local Jupyter

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print(" Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "comparison_results")
DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"   Output     -> {OUTPUT_DIR}")
print(f"   Data       -> {DATA_DIR}")
print(f"   Models     -> {SAVE_DIR}")


 Running on LOCAL JUPYTER
   Output     -> ./comparison_results
   Data       -> ./data
   Models     -> ./saved_models


In [2]:
# Prints installed package versions for NumPy, Pandas, OpenCV, and PyTorch, and confirms GPU availability and device name.
import torch, numpy as np, pandas as pd, cv2

print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"PyTorch    : {torch.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device     : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU -- inference will be slower.")


try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print(" ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!")
except ImportError as e:
    print(f" Missing package: {e}")
    print("  Run in terminal: pip install ultralytics seaborn tqdm kagglehub pyyaml")


NumPy      : 2.2.6
Pandas     : 2.3.3
OpenCV     : 4.13.0
PyTorch    : 2.10.0+cu128
Device     : cuda
GPU: NVIDIA GeForce RTX 3090
 ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!


In [3]:
# Downloads all three Kaggle pothole datasets through kagglehub
import kagglehub


print("Downloading datasets via kagglehub ...")
print("   (First run will open a browser to log in to Kaggle)")
print()

path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
print(f"chitholian     -> {path_1}")

path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
print(f"andrewmvd      -> {path_2}")

path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
print(f"ashishkumar    -> {path_3}")


DATASET_ROOTS = {
    "chitholian": path_1,
    "andrewmvd": path_2,
    "ashishkumar": path_3,
}

print(f"\nDataset roots: {DATASET_ROOTS}")


   (First run will open a browser to log in to Kaggle)

chitholian     -> /home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1
andrewmvd      -> /home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1
ashishkumar    -> /home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1

Dataset roots: {'chitholian': '/home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1', 'andrewmvd': '/home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1', 'ashishkumar': '/home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1'}


In [4]:
# Loads all shared imports and defines the fixed evaluation config, CLASS_NAMES, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the YOLO weight paths.
import time, json, glob, warnings, xml.etree.ElementTree as ET
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

warnings.filterwarnings("ignore")


CLASS_NAMES = ["pothole"]
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = 150

# YOLO weights
YOLO_WEIGHTS = {
    "YOLOv8m": "yolov8m.pt",
    "YOLOv10m": "yolov10m.pt",
    "YOLOv11m": "yolo11m.pt",
}

PALETTE = {
    "YOLOv8m": "#00d4ff",
    "YOLOv10m": "#3b82f6",
    "YOLOv11m": "#6366f1",
    "Faster R-CNN": "#f97316",
    "SSD-VGG16": "#ec4899",
}

print("Config loaded")
print(f"   Device      : {DEVICE}")
print(f"   Max images  : {MAX_IMAGES}")
print(f"   Conf thresh : {CONF_THRESH}")

Config loaded
   Device      : cuda
   Max images  : 150
   Conf thresh : 0.25
